# 🏋️ Train Your Own Flood Model — Google Colab *(2 of 2)*

**The training path:** auto-downloads the flood dataset, trains a fresh **MobileNet**
flood detector from scratch (or fine-tunes it), shows training curves + confusion matrix,
exports **Keras + TFLite** model files — and includes an optional **satellite-image
fine-tuning** section (FloodNet-style data or your own zip).

> 🔎 Only want to *detect* with the pre-trained model? Open **`01_Quick_Detection_Colab.ipynb`**.
> 🧩 Everything in one notebook → **`Flood_Detection_Colab.ipynb`**.

| # | Section | What happens |
|---|---------|--------------|
| 1 | Setup | GPU check + dependencies + seeds |
| 2 | Dataset | Auto-downloads the flood dataset (321 train / 81 valid / 72 test) |
| 3 | Training | MobileNet transfer learning, early stopping, curves |
| 4 | TFLite export | Mobile / edge-ready model file |
| 5 | Satellite fine-tune *(optional)* | Adapt to satellite / drone imagery |

> 💡 `Runtime ▸ Change runtime type ▸ T4 GPU` recommended — training takes a few minutes.

## 1️⃣ Setup & Environment

In [ ]:
# ============================================================================
# 1. SETUP — dependencies, GPU check, imports, seeds
# ============================================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

print('🔧 Installing dependencies...')
pip_install('gdown')

import os, shutil, zipfile, random, warnings, textwrap
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from IPython.display import display
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score, f1_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

LABELS    = ['Flooding', 'No Flooding']
IMG_SIZE  = (224, 224)
BATCH     = 32

gpu = tf.config.list_physical_devices('GPU')
print()
print('✅ Environment ready')
print('   TensorFlow :', tf.__version__)
print('   GPU        :', gpu[0].name if gpu else 'not available (CPU mode — still works, slightly slower)')

## 2️⃣ Get the Dataset *(automatic)*

Downloads the flood dataset (~52 MB) from the repo's Google Drive link and verifies the
`train / valid / test` splits.

In [ ]:
# ============================================================================
# 2. DATASET — download + extract (automatic)
# ============================================================================
DRIVE_DATA_ID = '1oT07-wVuYj_hByn4FqUtIj63Yg_fe7dP'


def get_dataset_dirs():
    '''Download + extract the flood dataset once; returns dict of split folders.'''
    zip_path = 'flood_dataset.zip'
    if not (os.path.isdir('data/train') or os.path.isdir('data/data/train')):
        import gdown
        print('⬇️  Downloading flood dataset (~52 MB)...')
        gdown.download(id=DRIVE_DATA_ID, output=zip_path, quiet=False)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall('.')
    base = 'data' if os.path.isdir('data/train') else 'data/data'
    return {s: os.path.join(base, s) for s in ['train', 'valid', 'test']}


dirs = get_dataset_dirs()
for split, folder in dirs.items():
    n = sum(len(fs) for _, _, fs in os.walk(folder))
    print(f'{split:6s}: {n} images  ({folder})')

## 3️⃣ Build & Train the Model

MobileNet trunk + `GlobalAveragePooling → Dropout(0.2) → Dense(2, softmax)` head,
top 23 layers trainable, Adam(1e-4), categorical cross-entropy, **EarlyStopping**.
Test accuracy typically lands around **95-98%**.

In [ ]:
# ============================================================================
# 3. TRAINING — build, train, evaluate (one click)
# ============================================================================
def build_flood_model(head_activation='softmax'):
    '''MobileNet trunk + GAP + Dropout + Dense(2) head.'''
    mobile = tf.keras.applications.MobileNet(weights='imagenet', include_top=False)
    x = mobile.get_layer('conv_pw_13_relu').output          # last ReLU map (1024 ch)
    x = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling2d')(x)
    x = tf.keras.layers.Dropout(0.2, name='dropout')(x)
    out = tf.keras.layers.Dense(2, activation=head_activation, name='dense')(x)
    return tf.keras.models.Model(mobile.input, out, name='flood_net')


MODELS_DIR = 'models'; os.makedirs(MODELS_DIR, exist_ok=True)


def train_fresh_model(epochs=15, verbose=2):
    '''Full automatic training pipeline with evaluation + curves.'''
    def make_ds(split, shuffle):
        ds = tf.keras.utils.image_dataset_from_directory(
            dirs[split], labels='inferred', label_mode='categorical',
            class_names=['flooding', 'normal'], image_size=IMG_SIZE,
            batch_size=BATCH, shuffle=shuffle, seed=SEED)
        return ds.map(lambda x, y: (tf.keras.applications.mobilenet.preprocess_input(x), y),
                      num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    train_ds, val_ds, test_ds = (make_ds('train', True), make_ds('valid', False),
                                 make_ds('test', False))

    model = build_flood_model('softmax')
    for layer in model.layers[:-23]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])

    print(f'🚀 Training ({epochs} epochs max, early-stopping on, '
          f'{sum(1 for l in model.layers if l.trainable)}/{len(model.layers)} layers trainable)...')
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=verbose,
                     callbacks=[tf.keras.callbacks.EarlyStopping(
                         monitor='val_accuracy', patience=3, restore_best_weights=True)])

    y_true = np.concatenate([np.argmax(y.numpy(), 1) for _, y in test_ds])
    y_pred = np.argmax(model.predict(test_ds, verbose=0), 1)
    print('\n📊 Test-set performance')
    print(f'   Accuracy : {accuracy_score(y_true, y_pred):.4f}')
    print(f'   Precision: {precision_score(y_true, y_pred, average="macro"):.4f}')
    print(f'   Recall   : {recall_score(y_true, y_pred, average="macro"):.4f}')
    print(f'   F1 Score : {f1_score(y_true, y_pred, average="macro"):.4f}')
    print(classification_report(y_true, y_pred, target_names=LABELS, digits=4))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(hist.history['accuracy'], 'o-', label='train')
    axes[0].plot(hist.history['val_accuracy'], 's-', label='validation')
    axes[0].set_title('Accuracy'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].plot(hist.history['loss'], 'o-', label='train')
    axes[1].plot(hist.history['val_loss'], 's-', label='validation')
    axes[1].set_title('Loss'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=.3)
    plt.show()

    model.save(os.path.join(MODELS_DIR, 'flood_model_trained.keras'))
    return model, hist


model, train_hist = train_fresh_model(epochs=15)

# confusion matrix on the test set
dirs = get_dataset_dirs()
test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    dirs['test'], labels='inferred', label_mode='categorical',
    class_names=['flooding', 'normal'], image_size=IMG_SIZE,
    batch_size=BATCH, shuffle=False)
y_true = np.concatenate([np.argmax(y.numpy(), 1) for _, y in test_ds_raw])
y_pred = np.argmax(model.predict(test_ds_raw.map(
    lambda x, y: (tf.keras.applications.mobilenet.preprocess_input(x), y)), verbose=0), 1)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(cm, cmap=plt.cm.Blues); fig.colorbar(im, ax=ax)
ax.set_xticks([0, 1], LABELS, rotation=45); ax.set_yticks([0, 1], LABELS)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Trained model — Confusion matrix (test)')
plt.show()

## 4️⃣ Export TFLite *(for Android / Raspberry Pi / edge devices)*

In [ ]:
# ============================================================================
# 4. TFLite export
# ============================================================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
TFLITE_PATH = os.path.join(MODELS_DIR, 'flood_model.tflite')
with open(TFLITE_PATH, 'wb') as fh:
    fh.write(tflite_model)
print(f'✅ Saved {TFLITE_PATH} ({os.path.getsize(TFLITE_PATH)/1e6:.1f} MB)')
print(f'✅ Keras model: {MODELS_DIR}/flood_model_trained.keras')

## 5️⃣ Satellite-Image Fine-Tuning *(optional — improves satellite accuracy)*

Same as the all-in-one notebook: upload a two-folder dataset zip (e.g. from
[FloodNet](https://github.com/BinaLab/FloodNet) or
[Sen1Floods11](https://github.com/cloudtostreet/Sen1Floods11)) and adapt the model at a
low learning rate.

In [ ]:
# ============================================================================
# 9. SATELLITE FINE-TUNING — upload a 2-folder dataset zip and adapt the model
# ============================================================================
from google.colab import files

SAT_ROOT = 'satellite_dataset'
choice = input('Type 1 + Enter to upload your satellite dataset .zip (or just Enter to skip): ').strip()

if choice != '1':
    print('ℹ️  Skipped. Re-run this cell any time to adapt the model to satellite imagery.')
else:
    shutil.rmtree(SAT_ROOT, ignore_errors=True)
    os.makedirs(SAT_ROOT, exist_ok=True)
    print('👆 Select your dataset .zip (two class folders inside)...')
    for name, data in files.upload().items():
        zpath = os.path.join(SAT_ROOT, os.path.basename(name))
        with open(zpath, 'wb') as fh:
            fh.write(data)
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(zpath) as z:
                z.extractall(SAT_ROOT)
    print('✅ extracted to', SAT_ROOT)

    # --- auto-detect the two class folders -------------------------------
    IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    class_dirs = []
    for root, dirs, fnames in os.walk(SAT_ROOT):
        if root == SAT_ROOT:
            continue
        if '__macosx' in root.lower():
            continue
        n_img = sum(1 for f in fnames if os.path.splitext(f)[1].lower() in IMG_EXT)
        if n_img >= 8:                                   # needs a usable number of images
            class_dirs.append(root)
    class_dirs = sorted(class_dirs)

    if len(class_dirs) != 2:
        raise RuntimeError(
            f'Expected exactly 2 class folders with images, found {len(class_dirs)}: '
            f'{class_dirs}. Please organise the zip as two folders, one per class.')

    sat_classes = [os.path.basename(c) for c in class_dirs]
    # decide which folder means "Flooding" (heuristic on folder names)
    FLOOD_HINTS = ('flood', 'water', 'inund', 'wet')
    first = sat_classes[0].lower()
    flood_idx = 0 if any(h in first for h in FLOOD_HINTS) else 1
    SAT_LABELS = ['Flooding', 'No Flooding'] if flood_idx == 0 else ['No Flooding', 'Flooding']
    print(f'📁 Class folders detected:')
    for i, c in enumerate(sat_classes):
        print(f'   class {i} -> {c!r}  =>  {SAT_LABELS[i]}')
    print('   (rename folders to include flood/water or dry/normal to control this mapping)')

    # --- datasets (auto 80/20 split) --------------------------------------
    common = dict(labels='inferred', label_mode='categorical',
                  class_names=sat_classes, image_size=IMG_SIZE, batch_size=BATCH,
                  seed=SEED, validation_split=0.2)
    sat_train = tf.keras.utils.image_dataset_from_directory(SAT_ROOT, subset='training', **common)
    sat_val   = tf.keras.utils.image_dataset_from_directory(SAT_ROOT, subset='validation', **common)
    sat_train = sat_train.map(
        lambda x, y: (tf.keras.applications.mobilenet.preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    sat_val = sat_val.map(
        lambda x, y: (tf.keras.applications.mobilenet.preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    # --- fine-tune FROM the current model (low LR keeps prior knowledge) ---
    sat_model = build_flood_model('softmax')
    sat_model.set_weights(model.get_weights())           # start from current best weights
    for layer in sat_model.layers[:-40]:                 # unfreeze a bit more for adaptation
        layer.trainable = False
    sat_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                      loss='categorical_crossentropy', metrics=['accuracy'])

    print(f'\n🛰️  Fine-tuning on satellite data '
          f'({sum(1 for l in sat_model.layers if l.trainable)}/{len(sat_model.layers)} layers trainable)...')
    sat_hist = sat_model.fit(sat_train, validation_data=sat_val, epochs=8, verbose=2,
                             callbacks=[tf.keras.callbacks.EarlyStopping(
                                 monitor='val_accuracy', patience=2,
                                 restore_best_weights=True)])

    # curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(sat_hist.history['accuracy'], 'o-', label='train')
    axes[0].plot(sat_hist.history['val_accuracy'], 's-', label='validation')
    axes[0].set_title('Satellite fine-tune - Accuracy'); axes[0].set_xlabel('epoch')
    axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].plot(sat_hist.history['loss'], 'o-', label='train')
    axes[1].plot(sat_hist.history['val_loss'], 's-', label='validation')
    axes[1].set_title('Satellite fine-tune - Loss'); axes[1].set_xlabel('epoch')
    axes[1].legend(); axes[1].grid(alpha=.3)
    plt.show()

    # confusion matrix on the validation split
    y_true = np.concatenate([np.argmax(y.numpy(), 1) for _, y in sat_val])
    y_pred = np.argmax(sat_model.predict(sat_val, verbose=0), 1)
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
    im = ax.imshow(cm, cmap=plt.cm.Blues); fig.colorbar(im, ax=ax)
    ax.set_xticks([0, 1], SAT_LABELS, rotation=45); ax.set_yticks([0, 1], SAT_LABELS)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Satellite model — Confusion matrix (validation)')
    plt.show()
    print(classification_report(y_true, y_pred, target_names=SAT_LABELS, digits=4))

    # save + switch the notebook over to the satellite model
    sat_model.save(os.path.join(MODELS_DIR, 'flood_model_satellite.keras'))
    model = sat_model
    model_source = 'satellite fine-tuned (this session)'
    LEGACY_SIGMOID = False
    print(f'✅ Saved {MODELS_DIR}/flood_model_satellite.keras')
    print('✅ model is now the satellite-tuned one — re-run Sections 4-7 to re-analyse your images.')

---
**Next step:** download `models/flood_model_trained.keras` (and `.tflite`) from the file
browser on the left, or run **`01_Quick_Detection_Colab.ipynb`** and re-run its Section 2
with your own weights.

*Credits: deeplizard, the flood-detection paper authors (arXiv:1909.00125), MobileNet.*